In [6]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.data.path.append("/usr/local/share/nltk_data")
%pip install sentence-transformers language-tool-python pysbd nltk


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ricardooodemacoo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/ricardooodemacoo/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You should consider upgrading via the '/usr/local/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ==========================
# 1. Imports
# ==========================
from sentence_transformers import SentenceTransformer, util
import pysbd
import language_tool_python
import nltk

# Ensure NLTK tokenizers are downloaded
nltk.download('punkt')

# ==========================
# 2. Load Embedding Model
# ==========================
print("Loading model...")
model = SentenceTransformer('sentence-transformers/roberta-base-nli-mean-tokens')
print("Model loaded successfully!")

# ==========================
# 3. Scoring Functions
# ==========================

def score_structure(essay):
    paragraphs = [p.strip() for p in essay.strip().split('\n\n') if p.strip()]
    transitions = ["however", "therefore", "moreover", "furthermore", "as a result", "in conclusion"]
    transition_count = sum(essay.lower().count(t) for t in transitions)
    if len(paragraphs) >= 3 and transition_count >= 2:
        return 90
    elif len(paragraphs) >= 2:
        return 75
    else:
        return 60

def score_grammar(essay):
    tool = language_tool_python.LanguageTool('en-US')
    matches = tool.check(essay)
    error_count = len(matches)
    length = len(essay.split())
    error_rate = error_count / length if length else 1
    if error_rate < 0.02:
        return 95
    elif error_rate < 0.05:
        return 85
    elif error_rate < 0.1:
        return 70
    else:
        return 50

def score_authenticity(essay):
    pronouns = sum(essay.lower().count(p) for p in [" i ", " me ", " my ", " mine ", " myself "])
    reflection_words = ["learned", "realized", "understood", "changed", "discovered", "grew"]
    reflection_hits = sum(essay.lower().count(w) for w in reflection_words)
    score = 50 + (pronouns * 3) + (reflection_hits * 5)
    return min(score, 95)

def score_emotional_impact(essay, model):
    emotional_templates = [
        "I was afraid", "I cried", "I felt proud", "I was ashamed",
        "I failed", "I overcame something difficult", "I was inspired", "I felt grateful"
    ]
    template_embeddings = model.encode(emotional_templates, convert_to_tensor=True)
    seg = pysbd.Segmenter(language="en", clean=True)
    sentences = seg.segment(essay)
    sentence_embeddings = model.encode(sentences, convert_to_tensor=True)
    hits = 0
    for i, sentence in enumerate(sentences):
        sim = util.cos_sim(sentence_embeddings[i], template_embeddings)
        if sim.max().item() > 0.65:
            hits += 1
    ratio = hits / len(sentences) if sentences else 0
    return round(ratio * 100, 2)

def score_reflection(essay, model):
    reflection_templates = [
        "I learned something important about myself.",
        "That experience changed me.",
        "I grew from that challenge.",
        "I gained a new perspective.",
        "I became more self-aware.",
        "It taught me a lesson.",
        "I realized something I hadn't before.",
        "This helped me understand myself better."
    ]
    template_embeddings = model.encode(reflection_templates, convert_to_tensor=True)
    seg = pysbd.Segmenter(language="en", clean=True)
    sentences = seg.segment(essay)
    sentence_embeddings = model.encode(sentences, convert_to_tensor=True)
    total_hits = 0
    for i, sentence in enumerate(sentences):
        sim_scores = util.cos_sim(sentence_embeddings[i], template_embeddings)
        if sim_scores.max().item() > 0.6:
            total_hits += 1
    return round((total_hits / len(sentences)) * 100, 2) if sentences else 0

def score_word_count(essay, max_words=650, min_words=250):
    count = len(essay.split())
    if min_words <= count <= max_words:
        return 95
    elif (count < min_words and count >= min_words - 50) or (count > max_words and count <= max_words + 100):
        return 80
    elif count < min_words - 50 or count > max_words + 150:
        return 50
    else:
        return 65

# ==========================
# 4. Essay Grading Function
# ==========================
def grade_full_essay(essay, model, max_words=650):
    reflection = score_reflection(essay, model)
    structure = score_structure(essay)
    grammar = score_grammar(essay)
    authenticity = score_authenticity(essay)
    emotion = score_emotional_impact(essay, model)
    word_score = score_word_count(essay, max_words=max_words)
    word_count = len(essay.split())

    overall = round(
        0.25 * reflection +
        0.2 * grammar +
        0.15 * authenticity +
        0.15 * structure +
        0.15 * emotion +
        0.10 * word_score,
        2
    )

    return {
        "Overall": overall,
        "Reflection": reflection,
        "Structure": structure,
        "Grammar": grammar,
        "Authenticity": authenticity,
        "Emotional Impact": emotion,
        "Word Count Score": word_score,
        "Actual Word Count": word_count
    }

# ==========================
# 5. Get User Input
# ==========================
print("\nPaste your essay below (type END on a new line to finish):")
lines = []
while True:
    line = input()
    if line.strip().upper() == "END":
        break
    lines.append(line)
essay_text = "\n".join(lines)

try:
    max_limit = int(input("Enter your maximum word count requirement (e.g. 650): "))
except ValueError:
    print("Invalid input. Using default max of 650.")
    max_limit = 650

# ==========================
# 6. Grade and Output
# ==========================
results = grade_full_essay(essay_text, model, max_words=max_limit)

print("\n=== Essay Evaluation ===")
for k, v in results.items():
    print(f"{k}: {v}%")


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ricardooodemacoo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading model...
Model loaded successfully!

Paste your essay below (type END on a new line to finish):
